In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Load dataset
df = pd.read_csv("RT_IOT2022.csv")
print("Initial dataset shape:", df.shape)
df.head()

In [ ]:
# Drop unneeded ID/metadata columns
cols_to_drop = ["Unnamed: 0", "Flow_ID", "Source_IP", "Destination_IP", "Timestamp", "id.orig_p", "id.resp_p"]
df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True, errors="ignore")

# Convert Attack_type to binary if desired, or handle rare classes
normal_traffic = ["Thing_Speak", "Wipro_bulb", "MQTT_Publish"]
if df["Attack_type"].dtype == object:
    df["Attack_type_binary"] = df["Attack_type"].apply(lambda x: 0 if x in normal_traffic else 1)

df.head()

In [ ]:
# Encode categorical features
label_encoders = {}
for col in df.select_dtypes(include=["object"]).columns:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le

# Target: binary classification (0 = Normal, 1 = Attack)
if "Attack_type_binary" in df.columns:
    y = df["Attack_type_binary"]
    X = df.drop(columns=["Attack_type", "Attack_type_binary"], errors="ignore")
else:
    y = df["Attack_type"]
    X = df.drop(columns=["Attack_type"], errors="ignore")

print("Features shape:", X.shape)
print("Target distribution:")
print(y.value_counts())

In [ ]:
# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

In [ ]:
# Model training and evaluation with XGBoost & LightGBM
xgb_model = XGBClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, eval_metric='logloss', random_state=42)
xgb_model.fit(X_train, y_train)
xgb_pred = xgb_model.predict(X_test)

print("=== XGBoost Results ====")
print("Accuracy:", accuracy_score(y_test, xgb_pred))
print("Macro-F1:", f1_score(y_test, xgb_pred, average="macro"))
print(classification_report(y_test, xgb_pred))

lgb_model = LGBMClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42, verbose=-1)
lgb_model.fit(X_train, y_train)
lgb_pred = lgb_model.predict(X_test)

print("\n=== LightGBM Results ====")
print("Accuracy:", accuracy_score(y_test, lgb_pred))
print("Macro-F1:", f1_score(y_test, lgb_pred, average="macro"))
print(classification_report(y_test, lgb_pred))